In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, roc_auc_score
from collections import Counter
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from collections import Counter

2025-03-29 23:24:16.501624: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 23:24:16.827193: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-29 23:24:17.006844: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743290657.218433   85810 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743290657.279056   85810 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743290658.421661   85810 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [6]:
df_original = pd.read_csv('Children Recode_final.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)
df.head()

X = df.drop(columns=['Malnurished'])
y = df['Malnurished']

# Train-test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state= 12)

# Columns to scale
columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
scaler = StandardScaler()

# Make copies of training and test sets
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale only selected columns
X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

# Apply SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", Counter(y_train))
print("After SMOTE:", Counter(y_train_sm))

Before SMOTE: Counter({0: 1110, 1: 569})
After SMOTE: Counter({0: 1110, 1: 1110})


**ANN Model**

In [8]:
# Build the ANN model
model = Sequential([
    Dense(64, input_dim=X_train_sm.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary output
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['Recall', 'Precision']
)

# Train the model
history = model.fit(
    X_train_sm, y_train_sm,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - Precision: 0.3674 - Recall: 0.2227 - loss: 0.6957 - val_Precision: 1.0000 - val_Recall: 0.0856 - val_loss: 0.9753
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Precision: 0.4096 - Recall: 0.1320 - loss: 0.6654 - val_Precision: 1.0000 - val_Recall: 0.1261 - val_loss: 0.9387
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Precision: 0.5470 - Recall: 0.1487 - loss: 0.6511 - val_Precision: 1.0000 - val_Recall: 0.1171 - val_loss: 0.9067
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Precision: 0.5136 - Recall: 0.1610 - loss: 0.6471 - val_Precision: 1.0000 - val_Recall: 0.1329 - val_loss: 0.9716
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Precision: 0.5286 - Recall: 0.1512 - loss: 0.6482 - val_Precision: 1.0000 - val_Recall: 0.1937 - val_loss: 0.8856
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - Precision: 0.5941 - Recall: 0.2254 - loss: 0.6548 - val_Precision: 1.0000 - val_Recall: 0.1171 - val_loss: 0.9507
Epoc

In [10]:

# Evaluate on test set
y_pred_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred))

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.68      0.78      0.73       371
           1       0.40      0.29      0.33       189

    accuracy                           0.61       560
   macro avg       0.54      0.53      0.53       560
weighted avg       0.59      0.61      0.60       560

